# SRM Melhoria de Processo

## Bibliotecas Usadas

In [71]:
import pandas as pd
import time
import numpy as np

In [72]:
 # Dados

periodo = 3
ano = 2026

## Montagem da Base N13P

In [73]:
#-#-#-# Inicio da contagem do tempo de execução #-#-#-#
inicio_total = time.time()
inicio_ciclo = time.time()
# Ciclo

colunas_base = ['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code','CD',
 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação', 'Marca']

colunas_periodos = [f'P{i:02d}-{ano}' for i in range(periodo, 14)]

colunas = colunas_base + colunas_periodos

tipos_colunas = {'EAN': str, 'COD_CLIENTE': str, 'Company Code': str, 'CD': str, 'SKU': str}

df_ciclo_n13 = pd.read_excel(f'../data/Arquivos/Ciclo_P{periodo:02d} N13P {ano} - envio.xlsx', header=3, usecols=colunas,
                             dtype=tipos_colunas,
                             engine='calamine')


# Clientes

tipos_colunas_clientes = {'COD_CLIENTE': str, 'COD REDE': str, 'COD SUBREDE': str, 'COND. PAG': str}

df_clientes = pd.read_excel('../data/Arquivos/BASE CLIENTES.xlsx', 
                            dtype=tipos_colunas_clientes,
                            engine='calamine')


# Produtos

tipos_colunas ={'EAN': str, 'SKU': str,
                                    'Ton/CDA': float, 
                                    'Unid/\nCX	': float, 
                                    'Hierarquia': str,
                                    'NCM': str,
                                    'kg/Un': float,
                                    'H05': str,
                                    'LSV': float}

df_produtos = pd.read_excel('../data/Arquivos/BASE PRODUTOS.xlsx', 
                            dtype=tipos_colunas,
                            engine='calamine')

df_produtos = df_produtos.rename(columns={'Unid/\nCX': 'Unid/CDA', 'kg/Un': 'kg/UN'})


# ZP 55

tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp55 = pd.read_excel('../data/Arquivos/ZP55.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp55['Cadastro'] = df_zp55['Cadastro'].round(2)

# ZP 54
tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp54 = pd.read_excel('../data/Arquivos/ZP54.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp54['Cadastro'] = df_zp54['Cadastro'].round(2)






#-#-#-# Colunas especificas #-#-#-#

# EAN Espelho

search_ean = df_produtos.set_index('EAN', drop=False)['EAN'].to_dict()
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN'].map(search_ean)

search_desc = df_produtos.set_index('Descrição', drop=False)['EAN'].to_dict()
secondary_search = df_ciclo_n13['Desc. SKU'].map(search_desc)
df_ciclo_n13['EAN Espelho'] = df_ciclo_n13['EAN Espelho'].fillna(secondary_search)


#  UF Origem

dicionario_uf = {
    'BR01': 'SP',
    'BR03': 'PE',
    'BR30': 'SP',
    'BR31': 'MG',
}

df_ciclo_n13['UF ORIGEM'] = df_ciclo_n13['CD'].map(dicionario_uf)

df_ciclo_n13[['CD', 'UF ORIGEM']].head(10)


# COD GP

dicionario_gp = {
    'ATACADO CASH & CARRY': 'AG',
    'GPA': 'AH',
    "SAM'S CLUB": 'AM',
    'GROCERY': 'AL',
    'ASSAI': 'AX',
    'Atacadão': 'TA',
    'DIST. MISTO': 'BI',
    'ESPECIALISTA DIRETO': 'AD',
    'DIA %': 'AF',
    'DIST. ALIMENTAR': 'AI',
    'DIST. ESPECIALISTA': 'AJ',
    'CENCOSUD': 'BJ',
    'CARREFOUR': 'AE',
    'ECOMMERCE': 'BO',
    'KA ESPECIALISTA': 'AQ',
    'PETZ': 'BL',
    'ATACADOS': 'AB',
    'MARTINS': 'BK',
    'COBASI': 'BM',
    'ATACADOS ESPECIAIS': 'BT',
    'CONVENIENCIAS': 'BP'
}

df_ciclo_n13['CÓD GP'] = df_ciclo_n13['GP'].map(dicionario_gp)

df_ciclo_n13['GP'] = df_ciclo_n13['GP'].str.strip()

# SKU

colunas_produtos = ['EAN', 'SKU', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 
                    'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

# 1. BASE EXATA (Com SKU)
df_prod_exato = df_produtos[colunas_produtos].drop_duplicates(subset=['EAN', 'SKU'], keep='first')

# 2. BASE DE RESGATE (Só EAN) - Tiramos o SKU para não dar conflito
df_prod_resgate = df_produtos[colunas_produtos].drop(columns=['SKU']).drop_duplicates(subset=['EAN'], keep='first')

# --- ETAPA 1: O CRUZAMENTO PERFEITO ---
df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_exato,
                        left_on=['EAN Espelho', 'SKU'],
                        right_on=['EAN', 'SKU'],
                        how='left')

# Limpa o EAN duplicado do primeiro cruzamento
df_ciclo_n13 = df_ciclo_n13.drop(columns=['EAN_y'], errors='ignore')
df_ciclo_n13 = df_ciclo_n13.rename(columns={'EAN_x': 'EAN'})

# --- ETAPA 2: O RESGATE DOS 51 MIL ÓRFÃOS ---
# Cruzamos de novo a mesma base, mas agora só pelo EAN e com a Base de Resgate.
# O "suffixes" coloca a palavra '_resgate' no nome das colunas novas para não misturar!
df_ciclo_n13 = pd.merge(df_ciclo_n13,
                        df_prod_resgate,
                        left_on='EAN Espelho',
                        right_on='EAN',
                        how='left',
                        suffixes=('', '_resgate'))

# Lista de colunas que precisamos tapar os buracos
colunas_preencher = ['Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV']

# O Loop Mágico: Se a coluna original estiver NaN, ele puxa o valor da coluna de resgate
for col in colunas_preencher:
    df_ciclo_n13[col] = df_ciclo_n13[col].fillna(df_ciclo_n13[f'{col}_resgate'])

# --- ETAPA 3: A FAXINA ---
# Apagamos as colunas temporárias de resgate que já usamos
colunas_lixo = [f'{col}_resgate' for col in colunas_preencher] + ['EAN_resgate']
df_ciclo_n13 = df_ciclo_n13.drop(columns=colunas_lixo, errors='ignore')

# Fazendo o teste final para comemorar!
linhas_lsv_nan = df_ciclo_n13['LSV'].isna().sum()
print(f"Número de linhas com LSV NaN após Resgate: {linhas_lsv_nan}")
print(f"Número de linhas do df_ciclo_n13 family: {len(df_ciclo_n13)}")

# Codigo Subrede

df_clientes_limpo = df_clientes[['COD_CLIENTE', 'COD SUBREDE', 'COND. PAG']].drop_duplicates(subset=['COD_CLIENTE'], keep='first')

df_ciclo_n13 = pd.merge(
    df_ciclo_n13,
    df_clientes_limpo, 
    on='COD_CLIENTE',
    how='left'
)

num_linhas = len(df_ciclo_n13)
print(f'Número de linhas do df_ciclo_n13: {num_linhas}')


fim_ciclo = time.time()
tempo_ciclo = fim_ciclo - inicio_ciclo
print(f'Tempo de execução: {tempo_ciclo:.2f} segundos')

Número de linhas com LSV NaN após Resgate: 0
Número de linhas do df_ciclo_n13 family: 351653
Número de linhas do df_ciclo_n13: 351653
Tempo de execução: 24.74 segundos


In [74]:
# df_ciclo_n13['Total Periodos'] = df_ciclo_n13[colunas_periodos].sum(axis=1)

# soma_periodos_por_coluna = df_ciclo_n13[colunas_periodos].sum()
# soma_total_geral = soma_periodos_por_coluna.sum()

# print("Soma por coluna dos períodos:")
# print(soma_periodos_por_coluna)
# print(f"\nSoma total geral: {soma_total_geral:.5f}")

# df_ciclo_n13[['Total Periodos'] + colunas_periodos].head()

# soma_lsv = df_ciclo_n13['LSV'].sum()
# print(f"Soma total LSV: {soma_lsv:.2f}")

## Calculos

### ZP55

In [75]:
inicio_zp55 = time.time()

df_zp55['CHAVE'] = df_zp55['CHAVE'].astype(str).str.strip()
dic_zp55 = df_zp55.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()

comp_code = df_ciclo_n13['Company Code'].astype(str).str.strip()
cod_cliente = df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip()
hierarquia_full = df_ciclo_n13['Hierarquia'].astype(str).str.strip() 
hierarquia_10 = hierarquia_full.str[:10]

# 3. CHAVES DE BUSCA (Agora espelhando o Excel com perfeição)
chave_1_full = comp_code + '_' + cod_cliente + '_' + hierarquia_full
chave_1_10 = comp_code + '_' + cod_cliente + '_' + hierarquia_10

# Limpando as outras colunas só por garantia também
cd_limpo = df_ciclo_n13['CD'].astype(str).str.strip()
uf_limpa = df_ciclo_n13['UF'].astype(str).str.strip()

chave_2 = cd_limpo + '_' + uf_limpa + '_' + df_ciclo_n13['Origem'].astype(str).str.strip()
chave_3 = cd_limpo + '_' + uf_limpa + '_' + df_ciclo_n13['NCM'].astype(str).str.strip()
chave_4 = cd_limpo + '_' + uf_limpa + '_' + hierarquia_10

# 4. AS BUSCAS
# Agora usamos a chave FULL primeiro, e se der erro (NaN), usamos a de 10.
df_ciclo_n13['CLIENTE'] = (chave_1_full.map(dic_zp55).fillna(chave_1_10.map(dic_zp55)) / 100)
df_ciclo_n13['CD + UF DESTINO + Importação'] = (chave_2.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + NCM'] = (chave_3.map(dic_zp55) / 100)
df_ciclo_n13['CD + UF DESTINO + H05'] = (chave_4.map(dic_zp55) / 100)

# 5. SELEÇÃO DA MELHOR OPÇÃO
df_ciclo_n13['ZP55'] = (df_ciclo_n13['CLIENTE']
                        .fillna(df_ciclo_n13['CD + UF DESTINO + Importação'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + NCM'])
                        .fillna(df_ciclo_n13['CD + UF DESTINO + H05'])
                        )

# Lembrando da nossa regra de ouro: Arredonda a ZP para 4 casas para não perder imposto!
df_ciclo_n13['ZP55'] = df_ciclo_n13['ZP55'].round(4)

fim_zp55 = time.time()
tempo_zp55 = fim_zp55 - inicio_zp55
print(f'Tempo de execução ZP55: {tempo_zp55:.2f} segundos')

# Print para você conferir a soma do CLIENTE na hora!
print(f"Soma CLIENTE atualizada: {df_ciclo_n13['CLIENTE'].sum():.2f}")

Tempo de execução ZP55: 1.75 segundos
Soma CLIENTE atualizada: 884.29


### ZP 54

In [76]:
inicio_zp54 = time.time()

dic_zp54 = df_zp54.set_index('CHAVE')['Cadastro'].to_dict()

hierarquia_12 = df_ciclo_n13['Hierarquia'].astype(str).str[:12]
hierarquia_10 = df_ciclo_n13['Hierarquia'].astype(str).str[:10]

# Chaves de Busca
chave_1_12 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + hierarquia_12
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + hierarquia_12
chave_3 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + hierarquia_12
chave_4 = df_ciclo_n13['Company Code'].astype(str) + '_' +df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + hierarquia_10

df_ciclo_n13['chave_4'] = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + '_' + df_ciclo_n13['UF'].astype(str) + '_' + hierarquia_10


# Busca
df_ciclo_n13['1. CLIENTE'] = (chave_1_12.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. REDE'] = (chave_2.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 6'] = (chave_3.map(dic_zp54) / 100).round(4)
df_ciclo_n13['1. GP UF HIER 5'] = (chave_4.map(dic_zp54) / 100).round(4)

# Seleção da melhor opção
df_ciclo_n13['ZP54'] = (df_ciclo_n13['1. CLIENTE']
                        .fillna(df_ciclo_n13['1. REDE'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 6'])
                        .fillna(df_ciclo_n13['1. GP UF HIER 5'])
                        )

# Arredondamento
df_ciclo_n13['ZP54'] = df_ciclo_n13['ZP54'].round(4)

fim_zp54 = time.time()
tempo_zp54 = fim_zp54 - inicio_zp54
print(f'Tempo de execução ZP54: {tempo_zp54:.2f} segundos')

Tempo de execução ZP54: 2.34 segundos


### GSVs

In [77]:
inicio_gsv = time.time()

# GSV/CDA

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['LSV'] * (1 + df_ciclo_n13['ZP55']) * (1 + df_ciclo_n13['ZP54'])

df_ciclo_n13['GSV/CDA'] = df_ciclo_n13['GSV/CDA'].round(4)

# df_ciclo_n13['ZP55'] = df_ciclo_n13['ZP55'].truncate(4)
# df_ciclo_n13['ZP54'] = df_ciclo_n13['ZP54'].truncate(4)

# GSV/TON

coluna_BA = df_ciclo_n13['GSV/CDA'] 
coluna_AA = df_ciclo_n13['kg/UN'] 
coluna_AO = df_ciclo_n13['Unid/CDA'] 

denominador = coluna_AA * coluna_AO

df_ciclo_n13['GSV/TON'] = np.where(
    (denominador == 0) | (denominador.isna()),
    np.nan,                                   
    (coluna_BA / denominador) * 1000           
)

df_ciclo_n13['GSV/TON'] = df_ciclo_n13['GSV/TON'].round(2)


fim_gsv = time.time()
tempo_gsv = fim_gsv - inicio_gsv
print(f'Tempo de execução GSV: {tempo_gsv:.2f} segundos')

Tempo de execução GSV: 0.02 segundos


### ZP53

In [78]:
### ZP 53

In [79]:
df_ciclo_n13[['LSV', 'ZP55', 'ZP54', 'kg/UN', 'Unid/CDA','GSV/CDA', 'GSV/TON']].head(10)


,LSV,ZP55,ZP54,kg/UN,Unid/CDA,GSV/CDA,GSV/TON
0,199.0,0.4826,-0.5039,0.9,10.0,146.3681,16263.12
1,199.0,0.4826,-0.5039,0.9,10.0,146.3681,16263.12
2,299.4,0.4826,-0.5655,2.7,6.0,192.8704,11905.58
3,199.0,0.4826,-0.5654,0.9,10.0,128.2233,14247.03
4,299.4,0.0867,-0.5130,2.7,6.0,158.4493,9780.82
5,199.0,0.0867,-0.5130,0.9,10.0,105.3154,11701.71
6,299.4,0.0867,-0.5130,2.7,6.0,158.4493,9780.82
7,199.0,0.0867,-0.5130,0.9,10.0,105.3154,11701.71
8,299.4,0.4826,-0.5039,2.7,6.0,220.2140,13593.46
9,199.0,0.4826,-0.5039,0.9,10.0,146.3681,16263.12


## Projeções

In [80]:
inicio_projecao = time.time()

for p in colunas_periodos:
    nome_coluna_projecao = f'GSV R$ {p} - {ano}'

    df_ciclo_n13[nome_coluna_projecao] = df_ciclo_n13[p] * df_ciclo_n13['GSV/TON']








fim_projecao = time.time()
tempo_projecao = fim_projecao - inicio_projecao

# primeiro_p = colunas_periodos[2]
# colunas_pra_validar = ['GSV/TON', primeiro_p, f'Projeção {primeiro_p} - {ano}']

# # print("\n--- Validação da Projeção ---")
# # print(df_ciclo_n13[colunas_pra_validar].head())

In [81]:
# codigo_alvo = '10460034'

# df_cliente_alvo = df_ciclo_n13[df_ciclo_n13['COD_CLIENTE'] == codigo_alvo]

# colunas_investigacao = [
#     'COD_CLIENTE', 'Hierarquia','LSV', 'ZP55', 'ZP54', 'GSV/CDA', 'kg/UN', 'Unid/CDA', 'GSV/TON', f'Projeção {primeiro_p} - {ano}'
# ]

# df_cliente_alvo[colunas_investigacao].head(20)

In [82]:
df_ciclo_n13['P04-2026'].head(10)

0   -0.000028
1   -0.000494
2   -0.014626
3   -0.002223
4   -0.045368
5   -0.050733
6   -0.064535
7   -0.072167
8   -0.025231
9   -0.028215
Name: P04-2026, dtype: float64

In [83]:
# df_ciclo_n13.to_excel('../data/Arquivos/Ciclo_P{:02d}_N13P_{}_processado.xlsx'.format(periodo, ano), index=False, engine='openpyxl')

# print(f'Arquivo salvo: ../data/Arquivos/Ciclo_P{periodo:02d}_N13P_{ano}_processado.xlsx')

In [84]:
df_ciclo_n13.columns

Index(['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code',
       'CD', 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU',
       'Classificação', 'Marca', 'P03-2026', 'P04-2026', 'P05-2026',
       'P06-2026', 'P07-2026', 'P08-2026', 'P09-2026', 'P10-2026', 'P11-2026',
       'P12-2026', 'P13-2026', 'EAN Espelho', 'UF ORIGEM', 'CÓD GP',
       'Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN',
       'Ton/CDA', 'Unid/CDA', 'LSV', 'COD SUBREDE', 'COND. PAG', 'CLIENTE',
       'CD + UF DESTINO + Importação', 'CD + UF DESTINO + NCM',
       'CD + UF DESTINO + H05', 'ZP55', 'chave_4', '1. CLIENTE', '1. REDE',
       '1. GP UF HIER 6', '1. GP UF HIER 5', 'ZP54', 'GSV/CDA', 'GSV/TON',
       'GSV R$ P03-2026 - 2026', 'GSV R$ P04-2026 - 2026',
       'GSV R$ P05-2026 - 2026', 'GSV R$ P06-2026 - 2026',
       'GSV R$ P07-2026 - 2026', 'GSV R$ P08-2026 - 2026',
       'GSV R$ P09-2026 - 2026', 'GSV R$ P10-2026 - 2026',
       'GSV R$ P11-2026 - 2026', 'GS

In [85]:
soma_cliente = df_ciclo_n13['CLIENTE'].sum()
soma_cd_uf_importacao = df_ciclo_n13['CD + UF DESTINO + Importação'].sum()
soma_cd_uf_ncm = df_ciclo_n13['CD + UF DESTINO + NCM'].sum()
soma_cd_uf_h05 = df_ciclo_n13['CD + UF DESTINO + H05'].sum()
soma_zp55 = df_ciclo_n13['ZP55'].sum()

print(f"Soma CLIENTE: {soma_cliente:.2f}")
print(f"Soma CD + UF DESTINO + Importação: {soma_cd_uf_importacao:.2f}")
print(f"Soma CD + UF DESTINO + NCM: {soma_cd_uf_ncm:.2f}")
print(f"Soma CD + UF DESTINO + H05: {soma_cd_uf_h05:.2f}")
print(f"Soma ZP55: {soma_zp55:.2f}")

Soma CLIENTE: 884.29
Soma CD + UF DESTINO + Importação: 260.08
Soma CD + UF DESTINO + NCM: 80090.07
Soma CD + UF DESTINO + H05: 2115.60
Soma ZP55: 81945.03


In [86]:
soma_1_cliente = df_ciclo_n13['1. CLIENTE'].sum()
soma_1_rede = df_ciclo_n13['1. REDE'].sum()
soma_1_gp_uf_hier_6 = df_ciclo_n13['1. GP UF HIER 6'].sum()
soma_1_gp_uf_hier_5 = df_ciclo_n13['1. GP UF HIER 5'].sum()
soma_zp54 = df_ciclo_n13['ZP54'].sum()

print(f"Soma 1. CLIENTE: {soma_1_cliente:.2f}")
print(f"Soma 1. REDE: {soma_1_rede:.2f}")
print(f"Soma 1. GP UF HIER 6: {soma_1_gp_uf_hier_6:.2f}")
print(f"Soma 1. GP UF HIER 5: {soma_1_gp_uf_hier_5:.2f}")
print(f"Soma ZP54: {soma_zp54:.2f}")

Soma 1. CLIENTE: -1451.77
Soma 1. REDE: -9243.19
Soma 1. GP UF HIER 6: -14834.58
Soma 1. GP UF HIER 5: -189778.57
Soma ZP54: -203569.19


In [87]:
soma_lsv = df_ciclo_n13['LSV'].sum()
soma_gsv_cda = df_ciclo_n13['GSV/CDA'].sum()
soma_gsv_ton = df_ciclo_n13['GSV/TON'].sum()

print(f"Soma LSV: {soma_lsv:.5f}")
print(f"Soma GSV/CDA: {soma_gsv_cda:.5f}")
print(f"Soma GSV/TON: {soma_gsv_ton:.5f}")


Soma LSV: 84231469.76000
Soma GSV/CDA: 42021537.69710
Soma GSV/TON: 9647773254.35000


In [88]:
colunas_gsv = ['GSV R$ P03-2026 - 2026', 'GSV R$ P04-2026 - 2026',
               'GSV R$ P05-2026 - 2026', 'GSV R$ P06-2026 - 2026',
               'GSV R$ P07-2026 - 2026', 'GSV R$ P08-2026 - 2026',
               'GSV R$ P09-2026 - 2026', 'GSV R$ P10-2026 - 2026',
               'GSV R$ P11-2026 - 2026', 'GSV R$ P12-2026 - 2026',
               'GSV R$ P13-2026 - 2026']

for col in colunas_gsv:
    soma = df_ciclo_n13[col].sum()
    print(f"Soma {col}: {soma:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))

soma_total_gsv = df_ciclo_n13[colunas_gsv].sum().sum()
print(f"\nSoma total de todos os GSV R$: {soma_total_gsv:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))


Soma GSV R$ P03-2026 - 2026: 186.602.148,89
Soma GSV R$ P04-2026 - 2026: 156.922.741,62
Soma GSV R$ P05-2026 - 2026: 179.173.823,26
Soma GSV R$ P06-2026 - 2026: 181.754.678,55
Soma GSV R$ P07-2026 - 2026: 190.343.092,09
Soma GSV R$ P08-2026 - 2026: 193.788.412,81
Soma GSV R$ P09-2026 - 2026: 196.410.608,83
Soma GSV R$ P10-2026 - 2026: 183.557.984,55
Soma GSV R$ P11-2026 - 2026: 186.228.283,75
Soma GSV R$ P12-2026 - 2026: 182.698.618,38
Soma GSV R$ P13-2026 - 2026: 116.598.927,77

Soma total de todos os GSV R$: 1.954.079.320,50


In [89]:
df_ciclo_n13['GSV/CDA'].head()

0    146.3681
1    146.3681
2    192.8704
3    128.2233
4    158.4493
Name: GSV/CDA, dtype: float64

## NIV

## Métricas

In [90]:
fim_total = time.time()
tempo_total = fim_total - inicio_total


print('Tempos:\n'
'Ciclo: {:.2f} segundos\n'
'ZP54: {:.2f} segundos\n'
'ZP55: {:.2f} segundos\n'
'GSV: {:.2f} segundos\n'
'Projeção: {:.2f} segundos\n'
'Total: {:.2f} segundos'
.format(tempo_ciclo, tempo_zp54, tempo_zp55, tempo_gsv, tempo_projecao, tempo_total))


excel = time.time()

# df_ciclo_n13.to_excel('../data/Arquivos/Ciclo_P{:02d}_N13P_{}_processado.xlsx'.format(periodo, ano), index=False, engine='openpyxl')

excel_fim = time.time()
tempo_excel = excel_fim - excel

print(f'Tempo para salvar Excel: {tempo_excel:.2f} segundos')

Tempos:
Ciclo: 24.74 segundos
ZP54: 2.34 segundos
ZP55: 1.75 segundos
GSV: 0.02 segundos
Projeção: 0.01 segundos
Total: 29.15 segundos
Tempo para salvar Excel: 0.00 segundos


In [91]:
# ZP 53
tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp53 = pd.read_excel('../data/Arquivos/ZP53.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp53['Cadastro'] = df_zp53['Cadastro'].round(2)

df_zp53['CHAVE'] = df_zp53['CHAVE'].astype(str).str.strip()


df_zp53.head()

,CHAVE,Cadastro
0,148_10261924_050101076001,6.65
1,148_17611214_050101076001,6.65
2,148_17633974_050101076001,4.31
3,148_17633974_050105400310,1.00
4,148_17633974_050105400320,1.00


In [92]:
print(df_ciclo_n13.columns.tolist())

['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code', 'CD', 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU', 'Classificação', 'Marca', 'P03-2026', 'P04-2026', 'P05-2026', 'P06-2026', 'P07-2026', 'P08-2026', 'P09-2026', 'P10-2026', 'P11-2026', 'P12-2026', 'P13-2026', 'EAN Espelho', 'UF ORIGEM', 'CÓD GP', 'Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN', 'Ton/CDA', 'Unid/CDA', 'LSV', 'COD SUBREDE', 'COND. PAG', 'CLIENTE', 'CD + UF DESTINO + Importação', 'CD + UF DESTINO + NCM', 'CD + UF DESTINO + H05', 'ZP55', 'chave_4', '1. CLIENTE', '1. REDE', '1. GP UF HIER 6', '1. GP UF HIER 5', 'ZP54', 'GSV/CDA', 'GSV/TON', 'GSV R$ P03-2026 - 2026', 'GSV R$ P04-2026 - 2026', 'GSV R$ P05-2026 - 2026', 'GSV R$ P06-2026 - 2026', 'GSV R$ P07-2026 - 2026', 'GSV R$ P08-2026 - 2026', 'GSV R$ P09-2026 - 2026', 'GSV R$ P10-2026 - 2026', 'GSV R$ P11-2026 - 2026', 'GSV R$ P12-2026 - 2026', 'GSV R$ P13-2026 - 2026']


In [93]:
# dic_zp53 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()

# chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str)
# chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str)
# chave_3 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + ' ' + df_ciclo_n13['UF'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str)
# chave_4 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['CÓD GP'].astype(str) + '_' + df_ciclo_n13['Hierarquia'].astype(str)[:10]

# df_ciclo_n13['EMISSOR'] = chave_1.map(dic_zp53) / 100
# df_ciclo_n13['REDE'] = chave_2.map(dic_zp53) / 100
# df_ciclo_n13['GP UF'] = chave_3.map(dic_zp53) / 100
# df_ciclo_n13['GP'] = chave_4.map(dic_zp53) / 100

# df_ciclo_n13['ZP53'] = (df_ciclo_n13['EMISSOR']
#                         .fillna(df_ciclo_n13['REDE'],)
#                         .fillna(df_ciclo_n13['GP UF'],)
#                         .fillna(df_ciclo_n13['GP'],)
#                         .fillna(0) # Se quiser colocar 0 no final caso todas as opções sejam NaN
#                     )

# df_ciclo_n13['ZP53'] = df_ciclo_n13['ZP53'].round(4)

# print(f"Soma total da coluna ZP53: {df_ciclo_n13['ZP53'].sum():.4f}")
# print(f"Soma total da coluna EMISSOR: {df_ciclo_n13['EMISSOR'].sum():.4f}")
# print(f"Soma total da coluna REDE: {df_ciclo_n13['REDE'].sum():.4f}")
# print(f"Soma total da coluna GP UF: {df_ciclo_n13['GP UF'].sum():.4f}")
# print(f"Soma total da coluna GP: {df_ciclo_n13['GP'].sum():.4f}")

# Criação do dicionário
dic_zp53 = df_zp53.drop_duplicates(subset=['CHAVE'], keep='first').set_index('CHAVE')['Cadastro'].to_dict()

# Blindagem contra espaços em branco (Opcional, mas salva vidas!)
comp_code = df_ciclo_n13['Company Code'].astype(str).str.strip()
cod_cliente = df_ciclo_n13['COD_CLIENTE'].astype(str).str.strip()
cod_subrede = df_ciclo_n13['COD SUBREDE'].astype(str).str.strip()
cod_gp = df_ciclo_n13['CÓD GP'].astype(str).str.strip()
uf = df_ciclo_n13['UF'].astype(str).str.strip()
hierarquia_full = df_ciclo_n13['Hierarquia'].astype(str).str.strip()

# Chaves de Busca (Com a correção do .str[:10])
chave_1 = comp_code + '_' + cod_cliente + '_' + hierarquia_full
chave_2 = comp_code + '_' + cod_subrede + '_' + hierarquia_full
chave_3 = comp_code + '_' + cod_gp + ' ' + uf + '_' + hierarquia_full
chave_4 = comp_code + '_' + cod_gp + '_' + hierarquia_full.str[:10] # <-- CORREÇÃO AQUI!

# Buscas
df_ciclo_n13['EMISSOR'] = chave_1.map(dic_zp53) / 100
df_ciclo_n13['REDE'] = chave_2.map(dic_zp53) / 100
df_ciclo_n13['GP UF'] = chave_3.map(dic_zp53) / 100
df_ciclo_n13['GP'] = chave_4.map(dic_zp53) / 100

# Cascata
df_ciclo_n13['ZP53'] = (df_ciclo_n13['EMISSOR']
                        .fillna(df_ciclo_n13['REDE']) # Tirei as vírgulas sobrando aqui
                        .fillna(df_ciclo_n13['GP UF'])
                        .fillna(df_ciclo_n13['GP'])
                        .fillna(0) # Se quiser colocar 0 no final caso todas as opções sejam NaN
                    )

df_ciclo_n13['ZP53'] = df_ciclo_n13['ZP53'].round(4)

print("Chaves 1: " + str(chave_1.head()))

print(f"Soma total da coluna ZP53: {df_ciclo_n13['ZP53'].sum():.4f}")
print(f"Soma total da coluna EMISSOR: {df_ciclo_n13['EMISSOR'].sum():.4f}")
print(f"Soma total da coluna REDE: {df_ciclo_n13['REDE'].sum():.4f}")
print(f"Soma total da coluna GP UF: {df_ciclo_n13['GP UF'].sum():.4f}")
print(f"Soma total da coluna GP: {df_ciclo_n13['GP'].sum():.4f}")

Chaves 1: 0    148_10460883_050101015527
1    148_10460883_050101015527
2    148_10460034_050101010427
3    148_10460034_050101015527
4    148_10261924_050101010427
dtype: str
Soma total da coluna ZP53: 655.3963
Soma total da coluna EMISSOR: 26.9796
Soma total da coluna REDE: 484.9934
Soma total da coluna GP UF: 0.0500
Soma total da coluna GP: 143.3733


In [94]:
df_ciclo_n13[['ZP53', 'EMISSOR', 'REDE', 'GP UF', 'GP']].head(10)

,ZP53,EMISSOR,REDE,GP UF,GP
0,0.0,NaN,NaN,NaN,NaN
1,0.0,NaN,NaN,NaN,NaN
2,0.0,NaN,NaN,NaN,NaN
3,0.0,NaN,NaN,NaN,NaN
4,0.0,NaN,NaN,NaN,NaN
5,0.0,NaN,NaN,NaN,NaN
6,0.0,NaN,NaN,NaN,NaN
7,0.0,NaN,NaN,NaN,NaN
8,0.0,NaN,NaN,NaN,NaN
9,0.0,NaN,NaN,NaN,NaN


# ZP 52

In [95]:
# ZP 52
tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp52 = pd.read_excel('../data/Arquivos/ZP52.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp52['Cadastro'] = df_zp52['Cadastro'].round(2)

df_zp52['CHAVE'] = df_zp52['CHAVE'].astype(str).str.strip()


df_zp52.head()

,CHAVE,Cadastro
0,148_40008217_05,-3.5
1,148_40008281_05,-3.5
2,148_40008282_05,-3.5
3,148_40008736_05,-5.6
4,148_40009202_05,-3.5


In [96]:
inicio_zp52 = time.time()

dic_zp52 = df_zp52.set_index('CHAVE')['Cadastro'].to_dict()

hierarquia_04 = df_ciclo_n13['Hierarquia'].astype(str).str[:8]
hierarquia_01 = df_ciclo_n13['Hierarquia'].astype(str).str[:2]

# Chaves de Busca
chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str) + '_' + hierarquia_04
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + hierarquia_01

df_ciclo_n13['chave teste 52'] = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str) + '_' + hierarquia_01


# Busca
df_ciclo_n13['H04'] = (chave_1.map(dic_zp52) / 100).round(4)
df_ciclo_n13['H01'] = (chave_2.map(dic_zp52) / 100).round(4)

# Seleção da melhor opção
df_ciclo_n13['ZP52'] = (df_ciclo_n13['H04']
                        .fillna(df_ciclo_n13['H01'])
                        .fillna(0)
                        )
                        

# Arredondamento
df_ciclo_n13['ZP52'] = df_ciclo_n13['ZP52'].round(4)

fim_zp52 = time.time()
tempo_zp52 = fim_zp52 - inicio_zp52
print(f'Tempo de execução ZP52: {tempo_zp52:.2f} segundos')

print(f"Soma total da coluna ZP52: {df_ciclo_n13['ZP52'].sum():.4f}")
print(f"Soma total da coluna H04: {df_ciclo_n13['H04'].sum():.4f}")
print(f"Soma total da coluna H01: {df_ciclo_n13['H01'].sum():.4f}")

Tempo de execução ZP52: 0.78 segundos
Soma total da coluna ZP52: -1562.0550
Soma total da coluna H04: 0.0000
Soma total da coluna H01: -1562.0550


In [97]:
df_ciclo_n13[['ZP52', 'chave teste 52',  'H04', 'H01']].head(10)

,ZP52,chave teste 52,H04,H01
0,0.00,148_41138608_05,NaN,NaN
1,0.00,148_41138608_05,NaN,NaN
2,-0.11,148_40131492_05,NaN,-0.11
3,-0.11,148_40131492_05,NaN,-0.11
4,0.00,148_41141691_05,NaN,NaN
5,0.00,148_41141691_05,NaN,NaN
6,0.00,148_41141691_05,NaN,NaN
7,0.00,148_41141691_05,NaN,NaN
8,0.00,148_41141691_05,NaN,NaN
9,0.00,148_41141691_05,NaN,NaN


# ZP 73

In [98]:
# ZP 73
tipos_colunas = {'CHAVE' : str, 'Cadastro': float}

colunas = ['CHAVE', 'Cadastro']

df_zp73 = pd.read_excel('../data/Arquivos/ZP73.xlsx', 
                        header=1,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp73['Cadastro'] = df_zp73['Cadastro'].round(2)

df_zp73['CHAVE'] = df_zp73['CHAVE'].astype(str).str.strip()


df_zp73.head()

,CHAVE,Cadastro
0,148_10149529,-7.35
1,148_10271748,-4.00
2,148_10307289,-3.50
3,148_10336702,-6.60
4,148_10374240,-7.35


In [99]:
inicio_zp73 = time.time()

dic_zp73 = df_zp73.set_index('CHAVE')['Cadastro'].to_dict()

# Chaves de Busca
chave_1 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD_CLIENTE'].astype(str)
chave_2 = df_ciclo_n13['Company Code'].astype(str) + '_' + df_ciclo_n13['COD SUBREDE'].astype(str)

# Seleção da melhor opção
df_ciclo_n13['ZP73'] = ((chave_1.map(dic_zp73) / 100).round(4)
                        .fillna((chave_2.map(dic_zp73) / 100).round(4))
                        .fillna(0)
                        )
                        

# Arredondamento
df_ciclo_n13['ZP73'] = df_ciclo_n13['ZP73'].round(4)

fim_zp73 = time.time()
tempo_zp73 = fim_zp73 - inicio_zp73
print(f'Tempo de execução ZP73: {tempo_zp73:.2f} segundos')

print(f"Soma total da coluna ZP73: {df_ciclo_n13['ZP73'].sum():.4f}")

Tempo de execução ZP73: 0.28 segundos
Soma total da coluna ZP73: -427.0213


# ZP 70

In [100]:
# ZP 70
tipos_colunas = {'CONDICAO DE PAGAMENTO' : str, 'Desconto': float}

colunas = ['CONDICAO DE PAGAMENTO', 'Desconto']

df_zp70 = pd.read_excel('../data/Arquivos/ZP70.xlsx', 
                        header=0,
                        usecols=colunas,
                        dtype=tipos_colunas,
                        engine='calamine')

df_zp70['Desconto'] = df_zp70['Desconto'].round(2)

df_zp70.head()

,CONDICAO DE PAGAMENTO,Desconto
0,Z112,-0.01
1,Z111,-0.01
2,Z113,-0.01
3,Z114,-0.00
4,Z115,-0.00


In [101]:
df_ciclo_n13.columns

Index(['Regional', 'GP', 'Gerente', 'Rede', 'COD_CLIENTE', 'Company Code',
       'CD', 'NOME_CLIENTE', 'UF', 'Região', 'EAN', 'SKU', 'Desc. SKU',
       'Classificação', 'Marca', 'P03-2026', 'P04-2026', 'P05-2026',
       'P06-2026', 'P07-2026', 'P08-2026', 'P09-2026', 'P10-2026', 'P11-2026',
       'P12-2026', 'P13-2026', 'EAN Espelho', 'UF ORIGEM', 'CÓD GP',
       'Family Price', 'Hierarquia', 'Class.', 'NCM', 'Origem', 'kg/UN',
       'Ton/CDA', 'Unid/CDA', 'LSV', 'COD SUBREDE', 'COND. PAG', 'CLIENTE',
       'CD + UF DESTINO + Importação', 'CD + UF DESTINO + NCM',
       'CD + UF DESTINO + H05', 'ZP55', 'chave_4', '1. CLIENTE', '1. REDE',
       '1. GP UF HIER 6', '1. GP UF HIER 5', 'ZP54', 'GSV/CDA', 'GSV/TON',
       'GSV R$ P03-2026 - 2026', 'GSV R$ P04-2026 - 2026',
       'GSV R$ P05-2026 - 2026', 'GSV R$ P06-2026 - 2026',
       'GSV R$ P07-2026 - 2026', 'GSV R$ P08-2026 - 2026',
       'GSV R$ P09-2026 - 2026', 'GSV R$ P10-2026 - 2026',
       'GSV R$ P11-2026 - 2026', 'GS

In [ ]:
inicio_zp70 = time.time()

dic_zp70 = df_zp70.set_index('CONDICAO DE PAGAMENTO')['Desconto'].to_dict()

# Chaves de Busca
chave_1 = df_ciclo_n13['COND. PAG'].astype(str)

# Seleção da melhor opção
df_ciclo_n13['ZP70'] = ((chave_1.map(dic_zp70)).round(4)
                        .fillna(0)
                        )
                        

# Arredondamento
df_ciclo_n13['ZP70'] = df_ciclo_n13['ZP70'].round(4)

fim_zp70 = time.time()
tempo_zp70 = fim_zp70 - inicio_zp70
print(f'Tempo de execução ZP70: {tempo_zp70:.2f} segundos')

print(f"Soma total da coluna ZP70: {df_ciclo_n13['ZP70'].sum():.4f}")

Tempo de execução ZP70: 0.02 segundos
Soma total da coluna ZP70: 1672.4000000000
